# 🧠 คำอธิบายและตัวอย่างการปฏิบัติการซัพพอร์ตเวกเตอร์แมชชีน (Support Vector Machines - SVM)

ยินดีต้อนรับสู่โน้ตบุ๊กประกอบการอธิบายเรื่อง **Support Vector Machines (SVM)**! ในโน้ตบุ๊กนี้เราจะ:
1. สร้างชุดข้อมูลจำลองแบบที่แบ่งแยกได้ด้วยเส้นตรง (Linearly Separable) และเทรน **Linear SVM** เพื่อหาพื้นผิวระนาบที่มีระยะขอบห่างมากที่สุด (Maximum Margin Separating Hyperplane)
2. พล็อตกราฟสังเกตจุดเวกเตอร์ค้ำจุน (Support Vectors), เส้นแบ่งขอบเขตการตัดสินใจ และเส้นระยะขอบ (Margin Lines)
3. สร้างชุดข้อมูลรูปวงกลมซ้อนแกนร่วมกัน เพื่อแสดงข้อจำกัดของการแบ่งตัวด้วยเส้นตรง และแก้ไขโดยประยุกต์ใช้งาน **กลวิธีเคอร์เนล RBF (RBF Kernel Trick)** เพื่อแบ่งพวกแยกจากกัน
4. พัฒนาแบบจำลอง **Linear SVM จากศูนย์ (from scratch)** โดยใช้กระบวนการ Gradient Descent ลงบนฟังก์ชันสูญเสียระยะขอบแบบยืดหยุ่น **(Soft-Margin Hinge Loss)**:
   $$J(\mathbf{w}, b) = \frac{1}{2} \|\mathbf{w}\|^2 + \frac{C}{m} \sum_{i=1}^{m} \max\left(0, 1 - y^{(i)}(\mathbf{w}^T \mathbf{x}^{(i)} + b)\right)$$
5. ฝึกสอนโมเดล SVM ที่เขียนเองและพล็อตแสดงขอบเขตและเส้นระยะขอบเพื่อตรวจสอบผลลัพธ์

เริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันก่อนครับ

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.datasets import make_blobs, make_circles
from sklearn.metrics import accuracy_score

# กำหนดค่า seed เพื่อให้ได้ผลลัพธ์การสุ่มเหมือนเดิมทุกครั้ง
np.random.seed(42)

## 1. แบบจำลองเชิงเส้น Linear SVM และเวกเตอร์ค้ำจุน (Support Vectors)

เราจะจำลองคุณลักษณะชุดข้อมูล 2 มิติที่สามารถแยกจากกันได้ด้วยเส้นตรง เพื่อแทนชั้นโลหะทรงกระบอก 2 คลาส: `polished-rod` (แกนเจียรผิวเรียบ: คลาส +1) และ `valve-stem` (ก้านวาล์ว: คลาส -1)

In [ ]:
# สุ่มสร้างจุดข้อมูล 40 จุดที่แยกออกจากการกันได้เด็ดขาด
X_lin, y_lin = make_blobs(n_samples=40, centers=2, random_state=6, cluster_std=0.60)
# แปลงป้ายคลาสเป้าหมายเป็น -1 และ +1 ตามรูปแบบข้อกำหนดทางคณิตศาสตร์ของ SVM
y_lin = np.where(y_lin == 0, -1, 1)

# พล็อตกราฟชุดข้อมูลจุดตัวอย่าง
plt.figure(figsize=(8, 5))
plt.scatter(X_lin[y_lin == 1, 0], X_lin[y_lin == 1, 1], color='blue', label='Class +1: Polished Rod', s=50)
plt.scatter(X_lin[y_lin == -1, 0], X_lin[y_lin == -1, 1], color='red', label='Class -1: Valve Stem', s=50)
plt.xlabel('Visual Feature 1')
plt.ylabel('Visual Feature 2')
plt.title('Separable Metal Rod Dataset')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

ต่อมา เราจะเทรนตัวจำแนกประเภท Linear SVM ด้วยไลบรารี `scikit-learn` โดยกำหนดให้ค่า $C$ มีขนาดสูง (Hard Margin หรือขอบเข้มงวด) เพื่อจำลองภาพขอบเขตการตัดสินใจ เส้นระยะขอบ และสังเกตจุดเวกเตอร์ค้ำจุน (Support Vectors) ที่โมเดลเลือกใช้ค้ำจูน

In [ ]:
# เทรนแบบจำลอง Linear SVM
svm_lin = SVC(kernel='linear', C=10.0)
svm_lin.fit(X_lin, y_lin)

# ดึงค่าน้ำหนักสัมประสิทธิ์และอคติ
w_sk = svm_lin.coef_[0]
b_sk = svm_lin.intercept_[0]
support_vecs = svm_lin.support_vectors_

# ฟังก์ชันสำหรับการวาดพื้นที่ขอบเขตการตัดสินใจและขอบเขตระยะขอบระยะทาง
def plot_svm_boundary(model, X, y, support_vectors=None):
    plt.scatter(X[y == 1, 0], X[y == 1, 1], color='blue', s=40, label='Class +1')
    plt.scatter(X[y == -1, 0], X[y == -1, 1], color='red', s=40, label='Class -1')
    
    ax = plt.gca()
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    
    xx = np.linspace(xlim[0], xlim[1], 30)
    yy = np.linspace(ylim[0], ylim[1], 30)
    YY, XX = np.meshgrid(yy, xx)
    xy = np.vstack([XX.ravel(), YY.ravel()]).T
    Z = model.decision_function(xy).reshape(XX.shape)
    
    ax.contour(XX, YY, Z, colors='k', levels=[-1, 0, 1], alpha=0.9, linestyles=['--', '-', '--'])
    
    if support_vectors is not None:
        ax.scatter(support_vectors[:, 0], support_vectors[:, 1], s=120,
                   linewidth=1.5, facecolors='none', edgecolors='black', label='Support Vectors')

plt.figure(figsize=(8, 5))
plot_svm_boundary(svm_lin, X_lin, y_lin, support_vecs)
plt.title('Linear SVM: Max Margin Classifier')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2. การจำแนกประเภทแบบไม่เป็นเชิงเส้น: กลวิธีเคอร์เนล (The Kernel Trick)

หากจุดกระจายตัวของชุดข้อมูลมีรูปทรงร่วมศูนย์กลางเป็นวงซ้อนกัน (ไม่สามารถแบ่งแยกด้วยเส้นตรงได้) เส้นพยากรณ์แบบตรงจะทำงานได้ย่ำแย่ เราจะลองสังเคราะห์ข้อมูลแบบวงกลมเปรียบเทียบ และประยุกต์ใช้งานคุณสมบัติแปลงมิติเคอร์เนลแบบ **Radial Basis Function (RBF)**

In [ ]:
# สุ่มสร้างชุดข้อมูลวงกลมซ้อนแกนร่วม
X_circle, y_circle = make_circles(n_samples=100, factor=0.3, noise=0.08, random_state=42)
y_circle = np.where(y_circle == 0, -1, 1)

# เทรนเปรียบเทียบระหว่าง Linear SVM และ RBF SVM
svm_rbf = SVC(kernel='rbf', C=1.0, gamma='scale')
svm_rbf.fit(X_circle, y_circle)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# พล็อตกราฟแสดงผลลัพธ์ของ Linear SVM บนวงกลมร่วมศูนย์กลาง
plt.sca(axes[0])
svm_linear_circle = SVC(kernel='linear', C=1.0)
svm_linear_circle.fit(X_circle, y_circle)
plot_svm_boundary(svm_linear_circle, X_circle, y_circle)
plt.title('Linear SVM Boundary (Underfitting)')

# Plot RBF Kernel SVM on concentric circles
plt.sca(axes[1])
plot_svm_boundary(svm_rbf, X_circle, y_circle)
plt.title('RBF Kernel SVM Boundary (Ideal Non-linear Separation)')

plt.show()

## 3. การเขียนแบบจำลอง SVM จากศูนย์ด้วยวิธีการหาอนุพันธ์ย่อย (Subgradient Descent SVM from Scratch)

เราจะพัฒนาแบบจำลอง Soft-Margin Linear SVM ขึ้นมาด้วยตนเอง โดยทำค่าฟังก์ชันต้นทุนต่อไปนี้ให้มีค่าน้อยที่สุด:
$$J(\mathbf{w}, b) = \frac{1}{2} \|\mathbf{w}\|^2 + \frac{C}{m} \sum_{i=1}^{m} \max\left(0, 1 - y^{(i)}(\mathbf{w}^T \mathbf{x}^{(i)} + b)\right)$$

โดยอ้างอิงวิธีเกรเดียนต์ย่อย (Subgradient Descent) บนแต่ละตัวอย่างข้อมูล $i$:
*   หากพบว่า $y^{(i)} (\mathbf{w}^T \mathbf{x}^{(i)} + b) \ge 1$ (ทำนายถูกคลาสและอยู่นอกบริเวณเขตระยะขอบ):
    $$\mathbf{w} \leftarrow \mathbf{w} - \alpha \cdot \frac{2 \mathbf{w}}{m}$$
*   ในกรณีอื่น (ทำนายผิดพลาด หรือจุดล้ำเข้าไปในบริเวณเขตระยะขอบ):
    $$\mathbf{w} \leftarrow \mathbf{w} - \alpha \cdot \left( \frac{2 \mathbf{w}}{m} - C \cdot y^{(i)} \mathbf{x}^{(i)} \right)$$
    $$b \leftarrow b + \alpha \cdot C \cdot y^{(i)}$$

มาลองลงมือเขียนโค้ดกันครับ!

In [ ]:
class CustomSVM:
    def __init__(self, C=1.0, learning_rate=0.001, epochs=1000):
        self.C = C
        self.lr = learning_rate
        self.epochs = epochs
        self.w = None
        self.b = 0.0

    def fit(self, X, y):
        m, n = X.shape
        self.w = np.zeros(n)
        self.b = 0.0
        
        for epoch in range(self.epochs):
            for idx, x_i in enumerate(X):
                condition = y[idx] * (np.dot(x_i, self.w) + self.b) >= 1
                if condition:
                    self.w -= self.lr * (2 * (1 / self.epochs) * self.w)
                else:
                    self.w -= self.lr * (2 * (1 / self.epochs) * self.w - self.C * y[idx] * x_i)
                    self.b += self.lr * self.C * y[idx]

    def decision_function(self, X):
        return np.dot(X, self.w) + self.b

    def predict(self, X):
        return np.sign(self.decision_function(X))

# เทรน Custom SVM บนชุดข้อมูลที่สามารถแยกเชิงเส้นได้
custom_svm = CustomSVM(C=10.0, learning_rate=0.01, epochs=800)
custom_svm.fit(X_lin, y_lin)

# ประเมินค่าความแม่นยำ
preds_custom = custom_svm.predict(X_lin)
print(f"Custom SVM Accuracy: {accuracy_score(y_lin, preds_custom) * 100:.2f}%")
print("Weights:", custom_svm.w, "| Bias:", custom_svm.b)

## 4. พล็อตแสดงขอบเขตการตัดสินใจของ Custom SVM

เราลองมาวาดภาพสังเกตขอบเขตการตัดสินใจและระยะขอบของแบบจำลอง Custom Scratch Linear SVM ที่เราเขียนขึ้นเองกันครับ

In [ ]:
plt.figure(figsize=(8, 5))
plot_svm_boundary(custom_svm, X_lin, y_lin)
plt.title('Custom Scratch Linear SVM Decision Boundary & Margins')
plt.grid(True, alpha=0.3)
plt.show()

## 💡 ความเชื่อมโยงสู่ Deep Learning และ YOLO
*   **ความสูญเสียแบบบานพับประตู (Hinge Loss):** ฟังก์ชันความสูญเสียระยะขอบที่ SVM ทำค่าให้ต่ำที่สุด (คือ $\max(0, 1 - y \cdot f(x))$) มีลักษณะความใกล้เคียงกับแนวคิดของฟังก์ชันความสูญเสียที่ใช้ประเมินความคลาดเคลื่อนในงานจำแนกประเภท หรือการถดถอยที่มีความเสถียร (เช่น Smooth L1 หรือ Huber Loss ที่นำมาใช้งานในขั้นตอนคำนวณตำแหน่งพิกัด Bounding Box Regression ของโมเดล YOLO)
*   **SVM บนคุณลักษณะที่สกัดจาก CNN (SVM on CNN Embeddings):** ในยุคเริ่มต้นของ Deep Learning ระบบจำพวก R-CNN จะใช้งานโครงข่ายประสาทสกัดฟีเจอร์รูปร่างเด่นออกมาก่อน จากนั้นจึงนำไปป้อนเข้าหัวแยกตัวแปรประเภทที่แบ่งคลาสด้วย SVM แต่ละประเภทสำหรับแยกประเภทวัตถุ แม้ว่าในปัจจุบันเราจะเปลี่ยนโครงสร้างมาฝึกสอนแบบเบ็ดเสร็จตั้งแต่ต้นจนจบ (End-to-End Training) ผ่านชั้น Fully Connected Classification Layer ทั้งหมดแล้ว ทว่าทฤษฎีการสร้างระยะขอบเขตที่ชัดเจนก็ยังเป็นกรอบคิดพื้นฐานที่ทรงคุณค่าอยู่